# HaloGS on Colab (T4)

Physically-grounded glow decomposition for 3D Gaussian Splatting on **real night scenes**.

`observed = clip( scene + k_APSF(q,T) ⊛ emission )` — the scene is parallax-consistent, the glow moves with the camera.

Runtime → Change runtime type → **T4 GPU**. Everything is stored under your Drive (`3DGS_Project`) so runs survive disconnects.

In [ ]:
import torch, subprocess
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/3DGS_Project'      # same layout as the thesis notebooks
import os; os.makedirs(ROOT, exist_ok=True)
%cd {ROOT}

## 1. Code: halo-gs + gaussian-splatting (CUDA rasterizer)

In [ ]:
%cd {ROOT}
![ -d halo-gs ] || git clone https://github.com/arpitpaliwal007/halo-gs
%cd {ROOT}/halo-gs
!git pull -q
!pip install -q -r requirements.txt
# official graphdeco repo (current API). Use `camenduru` as 2nd arg for the 2023 fork used in the earlier experiments.
!bash setup_gs.sh {ROOT}/gaussian-splatting official
os.environ['GS_ROOT'] = f'{ROOT}/gaussian-splatting'

In [ ]:
# COLMAP is only needed to triangulate points for LLFF scenes / run SfM for pose-less scenes
!apt-get -qq update && apt-get -qq install -y colmap > /dev/null
!colmap -h | head -3

## 2. Real glow scenes

Bilarf night captures (GlowGS uses `building`, `pondbike`, `strat`). ~7.8 GB zip, downloaded once to Drive.

In [ ]:
%cd {ROOT}/halo-gs
!bash data/download_bilarf.sh {ROOT}/data/raw
!ls {ROOT}/data/raw/bilarf

In [ ]:
# adjust the path if the archive's folder names differ from the printout above
SCENES = ['pondbike', 'building', 'strat']
BILARF = f'{ROOT}/data/raw/bilarf'
import glob
for s in SCENES:
    cand = glob.glob(f'{BILARF}/**/{s}', recursive=True)
    print(s, '->', cand)
    if cand:
        !python data/prepare_scene.py --scene {cand[0]} --out {ROOT}/data/scenes/bilarf/{s} --max_side 1600

## 3. Train: ours + the two baselines (same split / schedule / init)

7k iterations ≈ 6–8 min per method on a T4 at 1600 px; use 30k for final numbers.

In [ ]:
SCENE = 'pondbike'
ITERS = 7000
S = f'{ROOT}/data/scenes/bilarf/{SCENE}'
O = f'{ROOT}/out_halogs/{SCENE}'
for mode in ['halo', 'vanilla', 'masked']:
    !python train_glow.py -s {S} -m {O}/{mode} --mode {mode} --iterations {ITERS} --save_iterations {ITERS} \
        --data_device cpu --quiet --per_view_exposure

Per-view exposure is on because Bilarf adjusted exposure/ISO per shot; drop `--per_view_exposure` for RawNeRF scenes.

In [ ]:
for mode in ['halo', 'vanilla', 'masked']:
    !python render_glow.py -m {O}/{mode} --iteration {ITERS} --skip_train
    !python eval_glow.py   -m {O}/{mode} --iteration {ITERS} --save_masks
!python aggregate_results.py {ROOT}/out_halogs --iteration {ITERS} --md {O}/results.md
from IPython.display import Markdown; Markdown(open(f'{O}/results.md').read())

## 4. Look at the decomposition

In [ ]:
import json, matplotlib.pyplot as plt
from PIL import Image
rd = f'{O}/halo/renders/iteration_{ITERS}/test'
man = json.load(open(f'{rd}/manifest.json'))
v = man[0]['image_name']
fig, ax = plt.subplots(1, 5, figsize=(22, 5))
for a, (t, p) in zip(ax, [('observed', 'gt'), ('composite', 'render'), ('scene (glow-free)', 'scene'), ('glow x4', 'glow'), ('emission', 'emission')]):
    a.imshow(Image.open(f'{rd}/{p}/{v}.png')); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()
print(json.load(open(f'{O}/halo/config.json'))['kernel'], open(f'{O}/halo/log.csv').read().splitlines()[-1])

## 5. Ablations and the GlowGS few-shot protocol

In [ ]:
# kernel ablations (both / apsf / lens / radial / frozen / no sparsity / exposure)
# !bash scripts/ablations.sh {S} {ROOT}/out_abl/{SCENE} {ITERS}
# GlowGS protocol: 6 training views, rest test
# !python train_glow.py -s {S} -m {O}_glowgs/halo --mode halo --split glowgs --n_train_views 6 --iterations {ITERS} --save_iterations {ITERS} --data_device cpu --quiet

## 6. RawNeRF glow scenes (optional, large download)

In [ ]:
# !bash data/download_rawnerf.sh {ROOT}/data/raw
# for s in ['candlefiat','gardenlights','notchbush','parkstatue','stove','streetcorner']:
#     !python data/prepare_scene.py --scene {ROOT}/data/raw/rawnerf/scenes/{s} --out {ROOT}/data/scenes/rawnerf/{s}
#     !bash scripts/run_scene.sh {ROOT}/data/scenes/rawnerf/{s} {ROOT}/out_halogs/{s} {ITERS} --data_device cpu

## 7. Controlled synthetic glow (exact ground truth) from a well-lit posed scene, e.g. LOM `buu` normal-light

In [ ]:
# !python data/make_synthetic_glow.py --scene {ROOT}/datasets/buu_bright --out {ROOT}/data/scenes/synth/buu_glow --n_emitters 3 --T 1.5 --q 0.9 --gamma 6 --w_lens 0.4
# !bash scripts/run_scene.sh {ROOT}/data/scenes/synth/buu_glow {ROOT}/out_halogs/buu_glow {ITERS} --data_device cpu
# !python eval_glow.py -m {ROOT}/out_halogs/buu_glow/halo --iteration {ITERS} --clean_gt_dir {ROOT}/data/scenes/synth/buu_glow/gt_clean